In [4]:
# ============================================================================
# CÉLULA 1: PREPARAÇÃO DO AMBIENTE LOCAL (Dependências Obrigatórias)
# ============================================================================
import sys

print("Instalando e verificando dependências do ecossistema... Por favor, aguarde.")

# Lista completa de bibliotecas calibradas ao longo do projeto
!{sys.executable} -m pip install --quiet \
    torch \
    torchvision \
    diffusers>=0.26.0 \
    transformers \
    accelerate \
    safetensors \
    peft \
    datasets \
    torchmetrics \
    pillow \
    ipywidgets \
    gradio>=4.0.0 \
    deep-translator

print("\nAMBIENTE CONFIGURADO COM SUCESSO!")
print("Todas as dependências foram instaladas e estão prontas.")

Instalando e verificando dependências do ecossistema... Por favor, aguarde.

AMBIENTE CONFIGURADO COM SUCESSO!
Todas as dependências foram instaladas e estão prontas.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [ ]:
import torch
from diffusers import DiffusionPipeline

# ============================================================================
# 1. CONFIGURAÇÃO DE HARDWARE (Equilíbrio estável para CPU)
# ============================================================================
if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.float16
    variant = "fp16"
    print("Detectada GPU local!")

elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float32
    variant = None
    print("Detectado Apple Silicon!")

else:
    device = "cpu"
    dtype = torch.float32  # Voltamos para FP32 para a CPU calcular na velocidade máxima
    variant = None
    print("Modo CPU Otimizado Estável (FP32).")

# ============================================================================
# 2. CARREGA O SDXL COM STRATEGIES DE MEMÓRIA ATIVAS
# ============================================================================
print("Carregando Stable Diffusion XL Base 1.0...")

pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=dtype,
    variant=variant if variant else None,
    low_cpu_mem_usage=True,          
    use_safetensors=True
).to(device)

# TRAVA DE SEGURANÇA SE FOR CPU: 
if device == "cpu":
    print("Ativando fatiamento de atenção e otimização de canais...")
    pipe.enable_attention_slicing(slice_size="auto")  # Fatiamento dinâmico
    pipe.enable_vae_slicing()                         # Divide o decodificador de imagem em blocos

print("\nSUCESSO: Modelo SDXL carregado de forma estável!")

In [ ]:
# 3. Injeta os pesos do LoRA apontando para a pasta 'treinamentos'
lora_weights_path = "../treinamento/treino_local_pytorch_lora_weights.safetensors"

import os

if not os.path.exists(lora_weights_path):
    raise FileNotFoundError(
        f"Arquivo não encontrado em: '{lora_weights_path}':\n"
        "\n  1. Verifique se o nome da pasta é exatamente 'treinamento' (com ou sem maiúsculas) "
        "\n  2. Verifique se o arquivo pytorch_lora_weights.safetensors está na pasta."
    )

print(f"Injetando os pesos refinados do seu LoRA a partir de: {lora_weights_path}")

# Carrega os pesos passando o diretório correto e o nome do arquivo
pipe.load_lora_weights("../treinamento", weight_name="treino_local_pytorch_lora_weights.safetensors")
print("\nSUCESSO: Estilo 'estilo_azulejaria' acoplado e pronto para uso!")


In [ ]:
# ============================================================================
# 0. INSTALAÇÃO DAS BIBLIOTECAS UTILIZADAS NO PROJETO
# ============================================================================
import os
import sys
import random
import warnings
import gc
import tkinter as tk
from tkinter import ttk, messagebox
from deep_translator import GoogleTranslator

# Silencia avisos de logs para manter o console limpo
warnings.filterwarnings("ignore", category=UserWarning, module="transformers")
warnings.filterwarnings("ignore", category=UserWarning, module="peft")
warnings.filterwarnings("ignore", message=".*No LoRA keys.*")

# ============================================================================
# 1. CAPTURA DE ENTRADAS VIA POP-UP
# ============================================================================
def capturar_entradas_popup():
    root = tk.Tk()
    root.title("Ateliê de Azulejaria Luso-Brasileira")
    
    # Define proporção da pop-up
    root.geometry("550x260")
    root.resizable(False, False)
    root.attributes("-topmost", True)
    
    # Aplica estilização moderna nativa do sistema operacional
    style = ttk.Style()
    style.theme_use("vista" if os.name == "nt" else "default")
    
    respostas = {"objeto": "", "opcao": ""}

    # Container principal com espaçamento interno
    frame = ttk.Frame(root, padding="20")
    frame.pack(fill=tk.BOTH, expand=True)

    # Campo 1: Objeto em Português
    lbl_objeto = ttk.Label(frame, text="Digite o objeto desejado (em português):", font=("Segoe UI", 10))
    lbl_objeto.pack(anchor=tk.W, pady=(0, 2))
    
    txt_objeto = ttk.Entry(frame, font=("Segoe UI", 10))
    txt_objeto.pack(fill=tk.X, pady=(0, 12))
    txt_objeto.insert(0, "")
    txt_objeto.focus_set()

    # Campo 2: Estilo de Montagem
    lbl_estilo = ttk.Label(frame, text="Selecione o estilo da cerâmica:", font=("Segoe UI", 10))
    lbl_estilo.pack(anchor=tk.W, pady=(0, 2))
    
    combo_estilo = ttk.Combobox(frame, font=("Segoe UI", 10), state="readonly")
    combo_estilo['values'] = ("1 - Painel Histórico", "2 - Padrão Repetitivo")
    combo_estilo.current(0)
    combo_estilo.pack(fill=tk.X, pady=(0, 15))

    # Evento de validação e confirmação
    def confirmar():
        obj = txt_objeto.get().strip()
        if not obj:
            messagebox.showwarning("Aviso", "O objeto não pode ficar em branco!", parent=root)
            return
        
        respostas["objeto"] = obj
        respostas["opcao"] = combo_estilo.get()[0]
        root.destroy()

    # Botão de Envio Posicionado à Direita
    btn_confirmar = ttk.Button(frame, text="Gerar Azulejo", command=confirmar)
    btn_confirmar.pack(anchor=tk.E)

    # Tratamento caso o usuário feche a janela pelo 'X' do Windows
    def ao_fechar():
        print("OBSERVAÇÃO: Execução cancelada pelo usuário!")
        root.destroy()
        sys.exit()
    root.protocol("WM_DELETE_WINDOW", ao_fechar)

    root.mainloop()
    return respostas["objeto"], respostas["opcao"]

# Executa a interface gráfica calibrada
user_input_pt, opcao = capturar_entradas_popup()

# ============================================================================
# 2. MOTOR DE TRADUÇÃO NEURAL & PRINT DE AUDITORIA
# ============================================================================
try:
    # Força a tradução para o Inglês de padrão Americano
    user_input = GoogleTranslator(source='pt', target='en').translate(user_input_pt)
    user_input = user_input.lower().strip()
except Exception as e:
    print(f"ERRO: Falha na tradução automática ({e}). Usando termo original!")
    user_input = user_input_pt.lower().strip()

# Rastreabilidade dos pontos executados no processo
print("="*80)
print("                LOG DE AUDITORIA DA TRADUÇÃO INTERNA")
print(f" BR Entrada Original (PT-BR) ..: '{user_input_pt}'")
print(f" US Convertido para  (EN-US) ..: '{user_input}'")
print(f" Composição Selecionada .......:  {'Painel Histórico (Opção 1)' if opcao == '1' else 'Padrão Repetitivo (Opção 2)'}")
print("="*80)

# ============================================================================
# 3. CONFIGURAÇÃO DE PALETAS E PROMPTS (Seus Prompts Originais Congelados)
# ============================================================================
paletas_historicas = [
    "cobalt blue and white",
    "copper green and white",
    "yellow ochre and white",
    "terracotta and white"
]
cor_sorteada = random.choice(paletas_historicas)
print(f"\nPaleta Sorteada pelo Sistema: {cor_sorteada}.")

trigger = "estilo_azulejaria"
materialidade = "flat ceramic tiles, suttle grout lines, smooth glazed surface"
ornamentacao = "decorative corner motifs on each tile, traditional cantoneiras"
qualidade = "masterpiece, authentic antique look"

if opcao == "2":
    print(f"Configurando Padrão Repetitivo de Figura Avulsa.")
    prompt = (
        f"patchwork azulejo panel depicting a sharp detailed {user_input} alternating with plain white tiles, "
        f"{trigger}, {materialidade}, {cor_sorteada}, {qualidade}"
    )
    negative_prompt = (
        "deformed objects, fused tiles, broken grid, wavy lines, continuous landscape, photo, blurry, low quality"
    )
else:
    estilo_sorteado = random.choice(["continuo", "historico"])
    print(f"Configurando Painel Histórico/Contínuo \n  --> Sub-estilo selecionado: {estilo_sorteado}")

    if estilo_sorteado == "continuo":
        prompt = (
            f"A large {user_input} hand-painted on a continuous artwork painted across a ceramic tile mural, "
            f"subtle tile grout lines, elegant azulejo grid background, glossy glazed ceramic surface, "
            f"{trigger}, intricate blue and white patterns, {qualidade}"
        )
        negative_prompt = "modern photo, abstract mess, blurry, low quality"
    else:
        prompt = (
            f"historical azulejo panel depicting {user_input}, "
            f"{trigger}, {ornamentacao}, {materialidade}, {cor_sorteada}, {qualidade}"
        )
        negative_prompt = "modern photo, abstract mess, blurry, low quality"

print(f"\nPROMPT FINAL ENVIADO AO CLIP:\n'{prompt}'")
print("\n" + "="*80)
print("Renderizando imagem na CPU...")
print("="*80)

# ============================================================================
# 4. EXECUÇÃO DA INFERÊNCIA & SALVAMENTO AUTOMÁTICO SEGURO
# ============================================================================
# Força a limpeza profunda de resíduos de RAM na CPU antes de iniciar
gc.collect()

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,
    guidance_scale=7.5
).images[0]

output_dir = "../resultados" if "google.colab" not in sys.modules else "/content/imagens_generadas_estilo"
os.makedirs(output_dir, exist_ok=True)

clean_name = "".join([c for c in user_input if c.isalpha() or c.isspace()]).strip().replace(" ", "_")
tipo_sufixo = "mural" if opcao != "2" else "padrao"
output_filename = os.path.join(output_dir, f"resultado_{clean_name}_{tipo_sufixo}.png")

# SALVA NO DISCO IMEDIATAMENTE ANTES DE EXIBIR NA TELA
image.save(output_filename)
print(f"\nImagem salva com sucesso em:\n  -> {output_filename}\n")

# Tenta renderizar em tela apenas se a interface de vídeo local permitir
try:
    display(image)
except Exception as e:
    print(f"O display visual falhou devido ao pico de RAM gráfica, mas o seu arquivo está salvo e garantido na pasta resultados!")